# What happens if you put every variable into the model?

**Out of Sample** — [read the article](https://arthurnoo.pythonanywhere.com/every-variable-in-the-model)

This notebook reproduces every figure and every number in the article.

## Three ways to run it

| Where | What you need | Link |
| --- | --- | --- |
| **In your browser** | nothing at all | [Open in Colab](https://colab.research.google.com/github/Arthurnoo/out-of-sample/blob/main/notebooks/kitchen-sink.ipynb) |
| **In VS Code** | VS Code installed | [Clone and open](vscode://vscode.git/clone?url=https://github.com/Arthurnoo/out-of-sample) |
| **Anywhere else** | Python 3.10+ | you already have the file |

The next cell installs whatever is missing, so in every case you can simply
**Run All** and watch. The first run downloads prices and takes a minute or two.

## How to read it

Each section matches a section of the article. Run a cell, look at what it
prints, then read the note underneath explaining what to make of it.

**Section 0 holds every parameter you might want to change.** Everything after
it is written so that changing those values changes the whole notebook. That is
the point: the numbers below are mine, and the interesting question is what
happens to them when you move something.

Your numbers may differ slightly from the article if you run this later. The
data keeps growing, which is itself worth noticing.

## If you find a mistake

Tell me. There is a form at the bottom of the article, and a wrong number
corrected in public is worth more than a right one nobody checked.

In [ ]:
# Installs anything missing, then does nothing on later runs.
# This is what makes "Run All" work in Colab, in VS Code, or on a bare machine.
import importlib, subprocess, sys

for package in ['yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy']:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f'installing {package} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

print('ready')

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy import stats as st

pd.set_option('display.width', 130)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '#e4e9ee', 'axes.facecolor': 'white'})

## 0. The knobs

| Knob | What it does | Try |
| --- | --- | --- |
| `SYMBOL` | the market being predicted | `QQQ`, `AAPL`, `BTC-USD` |
| `START` | first day of data | `1995-01-01` for more history |
| `SPLIT` | where training stops and testing begins | move it and watch the story change |
| `HORIZON` | how many days ahead you predict | `5` for a weekly target |
| `N_SIM` | repetitions in the noise experiment | lower it to go faster |
| `RIDGE_GRID` | penalties tried | add `1e6` to push further |

The most instructive change is `SPLIT`. Move it to 2020 and the training period
swallows a decade of calm; move it to 2010 and the model is fitted on two
crashes. The conclusion survives, but the exact numbers move a lot, which is
itself the point of the piece.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  PARAMETERS — the only cell you need to edit
# ─────────────────────────────────────────────────────────────────────
SYMBOL  = 'SPY'
START   = '2000-01-01'
SPLIT   = '2015-01-01'      # everything before: training. after: never seen.
HORIZON = 1                 # predict the return this many days ahead

N_SIM      = 300            # repetitions of the pure-noise experiment
RIDGE_GRID = [0, 1, 10, 100, 1_000, 10_000, 100_000]

## 1. The result you should know by heart

Before touching market data. Take a target that is pure noise and variables that
are also pure noise, with no relationship whatsoever. Fit a regression. The R²
will not be zero.

Its expected value is exactly `k / n`: variables divided by observations.

In [ ]:
def ols(X, y):
    """Least squares via pseudo-inverse: stable even when columns are nearly
    collinear, which happens fast when you stack indicators built on the same
    prices."""
    X1 = np.column_stack([np.ones(len(X)), X])
    beta, *_ = np.linalg.lstsq(X1, y, rcond=None)
    pred = X1 @ beta
    ss_res = ((y - pred) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return beta, 1 - ss_res / ss_tot

rng = np.random.default_rng(0)
rows = []
for k in [1, 5, 10, 20, 50, 100]:
    n = 500
    r2s = [ols(rng.standard_normal((n, k)), rng.standard_normal(n))[1] for _ in range(N_SIM)]
    rows.append({'variables k': k, 'observations n': n,
                 'measured R2 %': np.mean(r2s) * 100, 'theory k/n %': k / n * 100})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:16.2f}'))

A hundred variables on five hundred days explain twenty percent of the variance
of something they have no relationship with.

**The question is never how high your R² is. It is how high it is compared to
k/n.**

## 2. Building the kitchen sink

Forty-four variables, all the kind anyone would build. Every one is computed
from information available on the day, and every one is shifted so it cannot see
the future. That shift is the only place where care is required.

In [ ]:
def rsi(close, n):
    d = close.diff()
    up = d.clip(lower=0).rolling(n).mean()
    down = (-d.clip(upper=0)).rolling(n).mean()
    return 100 - 100 / (1 + up / down.replace(0, np.nan))

def make_features(close, volume):
    r = close.pct_change()
    f = pd.DataFrame(index=close.index)
    for lag in range(1, 11):
        f[f'ret_lag{lag}'] = r.shift(lag)
    for w in (3, 5, 10, 21, 63, 126, 252):
        f[f'mom_{w}']     = close.pct_change(w).shift(1)
        f[f'vol_{w}']     = r.rolling(w).std().shift(1)
        f[f'dist_ma_{w}'] = (close / close.rolling(w).mean() - 1).shift(1)
    for w in (5, 21, 63):
        f[f'skew_{w}'] = r.rolling(w).skew().shift(1)
        f[f'kurt_{w}'] = r.rolling(w).kurt().shift(1)
    f['rsi_14'] = rsi(close, 14).shift(1)
    f['dow'] = close.index.dayofweek
    f['month'] = close.index.month
    f['day_of_month'] = close.index.day
    v = volume.replace(0, np.nan)
    f['vol_chg'] = v.pct_change().shift(1)
    for w in (5, 21):
        f[f'vol_rel_{w}'] = (v / v.rolling(w).mean()).shift(1)
    return f

h = yf.Ticker(SYMBOL).history(start=START, auto_adjust=True)
if h.index.tz is not None:
    h.index = h.index.tz_localize(None)
feat = make_features(h['Close'], h['Volume'])
target = h['Close'].pct_change(HORIZON).shift(-HORIZON)
df = pd.concat([feat, target.rename('y')], axis=1).dropna()
cols = [c for c in df.columns if c != 'y']

train, test = df[df.index < SPLIT], df[df.index >= SPLIT]
Xtr, ytr = train[cols].to_numpy(), train['y'].to_numpy()
Xte, yte = test[cols].to_numpy(), test['y'].to_numpy()
mu, sd = Xtr.mean(0), Xtr.std(0); sd[sd == 0] = 1
Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd

print(f"{len(cols)} variables")
print(f"training: {train.index[0].date()} to {train.index[-1].date()}  ({len(train)} days)")
print(f"testing : {test.index[0].date()} to {test.index[-1].date()}  ({len(test)} days)")

## 3. Adding variables, and watching it get worse

`r2_out` deserves a word. The denominator uses the mean of the **training**
data, not the test data. Using the test mean would hand the model information it
never had and flatter the result.

In [ ]:
def r2_out(beta, X, y, train_mean):
    pred = np.column_stack([np.ones(len(X)), X]) @ beta
    return 1 - ((y - pred) ** 2).sum() / ((y - train_mean) ** 2).sum()

rows = []
for k in [1, 2, 5, 10, 20, 30, len(cols)]:
    b, r2i = ols(Xtr[:, :k], ytr)
    rows.append({'variables': k, 'R2 in %': r2i * 100,
                 'R2 out %': r2_out(b, Xte[:, :k], yte, ytr.mean()) * 100,
                 'pure noise k/n %': k / len(ytr) * 100})
growth = pd.DataFrame(rows)
print(growth.to_string(index=False, float_format=lambda v: f'{v:17.3f}'))

fig, ax = plt.subplots(figsize=(7.4, 3.8))
ax.plot(growth['variables'], growth['R2 in %'], marker='o', color='#c83c50',
        label='on the data it was fitted to')
ax.plot(growth['variables'], growth['R2 out %'], marker='o', color='#0e8f5f',
        label='on data it never saw')
ax.plot(growth['variables'], growth['pure noise k/n %'], ls='--', color='#8b98a6',
        label='what pure noise gives')
ax.axhline(0, color='k', lw=1); ax.legend()
ax.set_xlabel('variables in the model'); ax.set_ylabel('R² (%)')
plt.show()

The red line can only go up: adding a variable can never worsen the fit on the
data being fitted, because the model can set its coefficient to zero. So
"my R² improved" is arithmetic, not evidence.

A negative out-of-sample R² means something specific: **worse than predicting the
average every day.**

## 4. What it costs in money

In [ ]:
def sharpe(x):
    x = np.asarray(x)
    return x.mean() / x.std(ddof=1) * np.sqrt(252 / HORIZON)

b, _ = ols(Xtr, ytr)
pred_in  = np.column_stack([np.ones(len(Xtr)), Xtr]) @ b
pred_out = np.column_stack([np.ones(len(Xte)), Xte]) @ b

print(f"trading the signal, in sample     : Sharpe {sharpe(np.sign(pred_in)*ytr):+.3f}"
      f"   hit rate {(np.sign(pred_in)==np.sign(ytr)).mean()*100:.1f}%")
print(f"trading the signal, out of sample : Sharpe {sharpe(np.sign(pred_out)*yte):+.3f}"
      f"   hit rate {(np.sign(pred_out)==np.sign(yte)).mean()*100:.1f}%")
print(f"buying and holding, same period   : Sharpe {sharpe(yte):+.3f}")

## 5. Why: bias, variance, and noise

```
error  =  bias²  +  variance  +  irreducible noise
```

Adding variables reduces bias and increases variance. With daily returns the
noise term dominates so completely that variance wins almost immediately.

The standard remedy is to shrink the coefficients toward zero: accept some bias
to remove a lot of variance.

In [ ]:
def ridge(X, y, lam):
    Xc, yc = X - X.mean(0), y - y.mean()
    beta = np.linalg.solve(Xc.T @ Xc + lam * np.eye(X.shape[1]), Xc.T @ yc)
    return np.concatenate([[y.mean() - X.mean(0) @ beta], beta])

rows = []
for lam in RIDGE_GRID:
    bb = ridge(Xtr, ytr, lam) if lam > 0 else b
    rows.append({'penalty': lam,
                 'R2 out %': r2_out(bb, Xte, yte, ytr.mean()) * 100,
                 'total size of coefficients': np.abs(bb[1:]).sum()})
rid = pd.DataFrame(rows)
print(rid.to_string(index=False, float_format=lambda v: f'{v:28.4f}'))

Read the last column. The best penalty shrinks the coefficients by a factor of a
thousand, and even then the out-of-sample R² is still slightly negative.

Regularisation did not rescue the model. It discovered that the best available
model was no model.

## 6. The last trap: testing many things

Drop the kitchen sink, look for the single best variable instead. That feels
more disciplined. It is a different route to the same mistake.

In [ ]:
rows = []
for c in cols:
    sl, ic, r, p, se = st.linregress(train[c].to_numpy(), ytr)
    rows.append({'feature': c, 't': sl / se if se else 0, 'p': p})
single = pd.DataFrame(rows)
single = single.reindex(single.t.abs().sort_values(ascending=False).index)
print(single.head(8).to_string(index=False, float_format=lambda v: f'{v:10.4f}'))

n_sig = (single.p < 0.05).sum()
bonf = 0.05 / len(cols)
print(f"\nsignificant at 5%          : {n_sig} of {len(cols)}")
print(f"expected by pure chance    : {0.05*len(cols):.1f}")
print(f"Bonferroni threshold       : p < {bonf:.5f}")
print(f"variables that survive it  : {(single.p < bonf).sum()}")

best = single.iloc[0]
sl, ic, *_ = st.linregress(train[best.feature].to_numpy(), ytr)
oos = sharpe(np.sign(sl * test[best.feature].to_numpy() + ic) * yte)
print(f"\nbest single variable: {best.feature} (t = {best.t:+.2f})")
print(f"its Sharpe out of sample      : {oos:+.3f}")
print(f"the 44-variable model's Sharpe: {sharpe(np.sign(pred_out)*yte):+.3f}")

One variable survives the correction, and on its own it beats the full model
out of sample. The kitchen sink contained a real signal and drowned it.

## 7. Things worth trying yourself

1. **Move `SPLIT`.** Try 2010 and 2020. Does the in-sample Sharpe stay around 1
   whatever you choose? It should, and that is the uncomfortable part.
2. **Set `HORIZON = 5` or `21`.** Longer horizons have better signal-to-noise.
   Does the out-of-sample R² turn positive anywhere?
3. **Shuffle the target** before fitting, so there is no relationship at all by
   construction, then compare the in-sample R² to k/n. It should land on it.
4. **Keep only the variables that survive Bonferroni** and fit that model. Does
   it beat the single best variable? Usually not, and that is worth sitting with.

Found something I got wrong? There is a form at the bottom of the article.